In [1]:
import numpy as np
import pandas as pd

# ==========================================================
# COSTANTI FISICHE
# ==========================================================
c = 2.99792458e10
G = 6.67430e-8
h = 6.62607015e-27
kB = 1.380649e-16
sigma = 5.670374419e-5
Msun = 1.98847e33

# ==========================================================
# PARAMETRI MODELLO SS
# ==========================================================
eta = 0.083
Nr = 2000
Nnu = 400
nu_min = 1e13
nu_max = 1e17

# ==========================================================
# FILE INPUT
# ==========================================================
file_linee = "Tab_completa_revisione.csv"
file_transizioni = "dati.asc"
file_mcmc = "risultati_MCMC.csv"

# ==========================================================
# FUNZIONI
# ==========================================================

def trova_colonna(df, possibili_nomi, obbligatoria=True):
    for nome in possibili_nomi:
        if nome in df.columns:
            return nome
    if obbligatoria:
        raise ValueError(f"Non trovo nessuna di queste colonne: {possibili_nomi}")
    return None


def costruisci_spettro_disco_SS(
    MBH_solar,
    mdot,
    eta=0.083,
    Nr=2000,
    Nnu=400,
    nu_min=1e13,
    nu_max=1e17
):
    MBH = MBH_solar * Msun

    Ledd = 1.3e38 * MBH_solar
    Ldisk = mdot * Ledd
    Mdot = Ldisk / (eta * c**2)

    Rs = 2 * G * MBH / c**2
    Rin = 3 * Rs
    Rout = 3000 * Rs

    R = np.logspace(np.log10(Rin), np.log10(Rout), Nr)

    f = 1.0 - np.sqrt(Rin / R)
    f = np.clip(f, 0.0, None)

    T = ((3 * G * MBH * Mdot) / (8 * np.pi * sigma * R**3) * f)**0.25

    nu = np.logspace(np.log10(nu_min), np.log10(nu_max), Nnu)

    Bnu = np.zeros((Nnu, Nr))

    for j in range(Nr):
        if T[j] <= 0:
            continue

        x = h * nu / (kB * T[j])
        x = np.clip(x, 1e-12, 700.0)

        Bnu[:, j] = (2 * h * nu**3 / c**2) / (np.exp(x) - 1.0)

    integrand = R[None, :] * Bnu
    I_nu = np.trapezoid(integrand, R, axis=1)

    Lnu = 4 * np.pi**2 * I_nu

    return nu, Lnu


def leggi_transizioni(file_transizioni):
    names, Eion, nu_ion_list = np.genfromtxt(
        file_transizioni,
        dtype=("U50", float, float),
        unpack=True
    )

    return pd.DataFrame({
        "line_name": names,
        "E_ion": Eion,
        "nu_ion": nu_ion_list
    })


def calcola_lion_per_transizioni(nu, Lnu, df_trans, nu_max=1e17):
    risultati = []

    for _, row in df_trans.iterrows():
        line_name = row["line_name"]
        E_ion = row["E_ion"]
        nu_ion = row["nu_ion"]

        if nu_ion >= nu_max:
            L_ion = 0.0
        else:
            mask = (nu >= nu_ion) & (nu <= nu_max)

            if np.sum(mask) < 2:
                L_ion = 0.0
            else:
                L_ion = np.trapezoid(Lnu[mask], nu[mask])

        risultati.append({
            "line_name": line_name,
            "E_ion": E_ion,
            "nu_ion": nu_ion,
            "L_ion": L_ion
        })

    return pd.DataFrame(risultati)


def estrai_linee_osservate(row, norm_columns):
    risultati = []

    for col in norm_columns:
        valore = row[col]

        if pd.isna(valore) or valore <= 0:
            continue

        line_name = col.replace("_norm", "")
        L_line = valore * 1e42

        risultati.append({
            "line_name": line_name,
            "L_line": L_line
        })

    return pd.DataFrame(risultati)


def analizza_blazar_SS(row, df_trans, norm_columns):
    nome_blazar = row["label"]
    MBH_solar = row["m_median_Msun"]
    mdot = row["mdot_median"]

    nu, Lnu = costruisci_spettro_disco_SS(
        MBH_solar=MBH_solar,
        mdot=mdot,
        eta=eta,
        Nr=Nr,
        Nnu=Nnu,
        nu_min=nu_min,
        nu_max=nu_max
    )

    df_lion = calcola_lion_per_transizioni(
        nu=nu,
        Lnu=Lnu,
        df_trans=df_trans,
        nu_max=nu_max
    )

    df_linee = estrai_linee_osservate(row, norm_columns)

    if df_linee.empty:
        return pd.DataFrame()

    df_conf = df_linee.merge(
        df_lion,
        on="line_name",
        how="left"
    )

    df_conf["covering_factor"] = df_conf["L_line"] / df_conf["L_ion"]

    df_conf.loc[
        (df_conf["L_ion"].isna()) | (df_conf["L_ion"] <= 0),
        "covering_factor"
    ] = np.nan

    df_conf["label"] = nome_blazar

    return df_conf[["label", "line_name", "covering_factor"]]


# ==========================================================
# LETTURA FILE
# ==========================================================

df_tab = pd.read_csv(file_linee)
df_trans = leggi_transizioni(file_transizioni)
df_mcmc = pd.read_csv(file_mcmc)

col_name_tab = trova_colonna(
    df_tab,
    ["label", "file_name", "name", "nome", "blazar"]
)

df_tab = df_tab.rename(columns={col_name_tab: "label_originale"}).copy()
df_mcmc = df_mcmc.rename(columns={"Blazar": "label"}).copy()

# ==========================================================
# SISTEMO I NOMI DEI BLAZAR
# ==========================================================

df_tab["label_originale"] = df_tab["label_originale"].astype(str).str.strip()
df_mcmc["label"] = df_mcmc["label"].astype(str).str.strip()

df_tab["label"] = df_tab["label_originale"].str.extract(r"(J\d+)", expand=False)
df_mcmc["label"] = df_mcmc["label"].str.extract(r"(J\d+)", expand=False)

norm_columns = [col for col in df_tab.columns if col.endswith("_norm")]

# ==========================================================
# MERGE CON RISULTATI MCMC
# ==========================================================

df_all = df_tab.merge(
    df_mcmc[["label", "m_median_Msun", "mdot_median"]],
    on="label",
    how="inner"
)

if len(df_all) == 0:
    raise ValueError("Il merge non ha trovato blazar in comune.")


# ==========================================================
# CALCOLO CF PER TUTTI I BLAZAR
# ==========================================================

tutti_risultati = []

for i in range(len(df_all)):
    row = df_all.iloc[i]

    df_conf = analizza_blazar_SS(
        row=row,
        df_trans=df_trans,
        norm_columns=norm_columns
    )

    if not df_conf.empty:
        tutti_risultati.append(df_conf)

if len(tutti_risultati) == 0:
    raise ValueError("Nessun covering factor calcolato.")

df_cf_SS_mcmc = pd.concat(tutti_risultati, ignore_index=True)


# ==========================================================
# TABELLA FINALE: BLAZAR + CF PER LINEA + MEDIA
# ==========================================================

tabella_finale = df_cf_SS_mcmc.pivot_table(
    index="label",
    columns="line_name",
    values="covering_factor",
    aggfunc="first"
)

tabella_finale["Cf_mean"] = tabella_finale.mean(axis=1, skipna=True)

# ordine colonne più leggibile
ordine_linee = ["CIII", "CIV", "MgII", "Ha", "Hb", "Hg", "Hd"]
colonne_presenti = [c for c in ordine_linee if c in tabella_finale.columns]

tabella_finale = tabella_finale[
    colonne_presenti + ["Cf_mean"]
]

tabella_finale = tabella_finale.reset_index()

# ==========================================================
# STAMPA SOLO LA TABELLA FINALE
# ==========================================================

display(tabella_finale)



line_name,label,CIII,CIV,MgII,Ha,Hb,Hg,Hd,Cf_mean
0,J0021,NaN,NaN,NaN,NaN,0.002552,0.000822,0.000643,1.338898e-03
1,J0050,NaN,NaN,NaN,0.003110,0.000690,0.000598,0.000113,1.127846e-03
2,J0059,NaN,NaN,0.338154,NaN,NaN,0.144589,0.066265,1.830026e-01
3,J0133,NaN,NaN,0.200552,NaN,NaN,NaN,0.012532,1.065424e-01
4,J0508,7.986748e-02,5.234301e-01,NaN,NaN,NaN,NaN,NaN,3.016488e-01
5,J0922,NaN,NaN,0.834362,NaN,0.001645,0.001033,0.003088,2.100321e-01
6,J1123,NaN,NaN,NaN,0.000871,0.002140,0.000967,0.000354,1.082789e-03
7,J1329,NaN,NaN,0.210505,NaN,NaN,0.001312,0.005121,7.231286e-02
8,J152422,1.623454e+14,NaN,1.730187,NaN,NaN,NaN,0.001146,5.411512e+13
9,J170108,7.804100e+00,2.325991e+05,0.001826,NaN,NaN,NaN,NaN,7.753564e+04


NOTO CHE i blazar con cf migliori (circa 10^-1) sono quelli che dai risultati_MCMC (pwp) sono di tipo SS